# 04 · OpenVLA-OFT 독립 환경 1-step FFT·재로딩
01에서 준비한 동일 canonical split/normalization을 사용합니다. AVA 환경과 Transformers fork를
공유하지 않습니다. LoRA가 아닌 전체 파라미터 FFT이며, 본격 학습 전 1 optimizer step과 저장·재로딩만 검사합니다.
A100 40 GB에서도 OOM 가능성이 있으므로 실제 peak VRAM과 Drive 여유 공간을 먼저 확인하십시오.

In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib, json, os, shutil, subprocess, sys
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/VLADE')  # 세 노트북에서 동일한 저장 폴더를 지정하십시오.
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = ROOT  # src/scripts/configs/experiments/third_party가 바로 들어 있는 실제 프로젝트 루트
EXPECTED_MANIFEST_SHA256 = "db79038305323f4eb5328e5986c87f889fd0e74158ee31fc9de8f944bb418d9e"
def stage_source(project_root: Path, scratch_root: Path, expected_manifest_sha256: str) -> Path:
    """Verify Drive source files and stage them without copying data or weights."""
    project_root = project_root.resolve()
    manifest_relative = Path("experiments/manifests/colab_source.json")
    manifest_path = project_root / manifest_relative
    if not manifest_path.is_file():
        raise FileNotFoundError("Drive의 VLADE 폴더에 src/scripts/configs/experiments/third_party 일반 파일 구조를 먼저 복사하십시오: " + str(manifest_path))
    manifest_bytes = manifest_path.read_bytes()
    if hashlib.sha256(manifest_bytes).hexdigest() != expected_manifest_sha256:
        raise ValueError("노트북과 colab_source.json 버전이 다릅니다. 같은 버전의 일반 프로젝트 파일을 사용하십시오.")
    manifest = json.loads(manifest_bytes)
    code = scratch_root / ("vlade_colab_" + expected_manifest_sha256[:12])
    code = code.resolve()
    if code.is_relative_to(project_root) or project_root.is_relative_to(code):
        raise ValueError("실행 소스 임시 폴더와 Drive 프로젝트 폴더는 서로 겹칠 수 없습니다.")
    for relative, expected in manifest["files_sha256"].items():
        path = Path(relative)
        if path.is_absolute() or ".." in path.parts or "\\" in relative or ":" in relative:
            raise ValueError("잘못된 소스 상대 경로: " + relative)
        source = project_root / path
        target = code / path
        if not source.resolve().is_relative_to(project_root) or not target.resolve().is_relative_to(code):
            raise ValueError("프로젝트 밖의 소스 경로: " + relative)
        if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
            raise ValueError("Drive 소스가 없거나 변경되었습니다: " + relative)
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.is_file() or hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            shutil.copyfile(source, target)
        if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
            raise ValueError("복사된 소스 무결성 불일치: " + relative)
    # Publish the manifest only after every listed source file has been verified.
    staged_manifest = code / manifest_relative
    staged_manifest.parent.mkdir(parents=True, exist_ok=True)
    temporary_manifest = staged_manifest.with_suffix(".json.tmp")
    temporary_manifest.write_bytes(manifest_bytes)
    temporary_manifest.replace(staged_manifest)
    return code

CODE = stage_source(PROJECT_ROOT, Path('/content'), EXPECTED_MANIFEST_SHA256)
for module_name, module in list(sys.modules.items()):
    if module_name == 'vlade' or module_name.startswith('vlade.'):
        filename = getattr(module, '__file__', None)
        if filename and not Path(filename).resolve().is_relative_to(CODE.resolve()):
            raise RuntimeError('다른 버전의 VLADE가 메모리에 있습니다. Colab 런타임을 다시 시작하십시오.')
sys.path.insert(0, str(CODE / 'src'))
print('Drive:', ROOT, '\n실행 소스:', CODE)
ENV = os.environ.copy()
ENV.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled', TF_CPP_MIN_LOG_LEVEL='2',
           HF_HOME=str(ROOT / 'hf_cache'))
def run(args, env=None):
    process = subprocess.Popen([str(x) for x in args], cwd=CODE, env=env or ENV,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError('실행에 실패했습니다. 위 오류를 확인하십시오. 후속 학습은 시작하지 않습니다.')
if shutil.which('uv') is None:
    run([sys.executable, '-m', 'pip', 'install', 'uv'])
UV = shutil.which('uv')
assert UV
def environment(name, version, packages):
    directory = Path('/content') / name
    python = directory / 'bin/python'
    if not python.is_file():
        run([UV, 'venv', '--python', version, directory])
    run([UV, 'pip', 'install', '--python', python, *packages])
    return python
def completed_checkpoints(directory):
    return sorted(path for path in directory.glob('step-*')
                  if (path / 'COMPLETE').is_file() and (path / 'trainer_state.pt').is_file())

In [ ]:
import tomllib
upstream = CODE / 'third_party/openvla_oft'
dependencies = tomllib.loads((upstream / 'pyproject.toml').read_text())['project']['dependencies']
dependencies = [dep for dep in dependencies if not dep.startswith(('transformers @', 'dlimp @', 'huggingface_hub', 'accelerate'))]
dependencies += [
    'transformers @ git+https://github.com/moojink/transformers-openvla-oft.git@bc339d9ad707454c0c115970db43c260067c61ab',
    'dlimp @ git+https://github.com/moojink/dlimp_openvla.git@040105d256bd28866cc6620621a3d5f7b6b91b46',
    'numpy==1.26.4', 'tensorflow-metadata==1.15.0', 'bitsandbytes==0.43.3',
    'huggingface_hub==0.36.2', 'accelerate==1.15.0']
OFT_PY = environment('vlade_oft_fft_py311', '3.11', dependencies)
run([UV, 'pip', 'install', '--python', OFT_PY, '--no-deps', '-e', upstream])
run([OFT_PY, '-c', "import torch,transformers,tensorflow_datasets; assert torch.cuda.is_available(); assert torch.cuda.is_bf16_supported(); print(torch.__version__,transformers.__version__,torch.cuda.get_device_name(0))"])
(ROOT / 'oft_environment_freeze.txt').write_text(subprocess.check_output(
    [str(UV), 'pip', 'freeze', '--python', str(OFT_PY)], text=True))

In [ ]:
PREPARED = ROOT / 'data/processed/libero_plus_episode80'
from vlade.data.episode_data import EpisodeData
data = EpisodeData(PREPARED)
assert data.manifest['counts'] == {'train': 11477, 'validation': 1428, 'test': 1442}
assert data.manifest['canonical_manifest_sha256'] == '487d2be1039ba6a4facc3ba44785ba5159245dfe53462a1d5a41ba8230446e80'
config = json.loads((CODE / 'configs/experiments/libero_plus_fft.json').read_text())
output = ROOT / 'runs/openvla_oft_fft/oft-one-step-seed42'
config['fft_defaults'].update(
    data_root=str(ROOT / 'data/processed/libero_plus_tfds'), prepared_data_root=str(PREPARED),
    image_aug=False, batch_size=1, grad_accumulation_steps=1, max_steps=1, save_steps=1,
    save_optimizer_state=True, eval_steps=0)
config['fft_runs']['openvla_oft'].update(
    checkpoint=str(ROOT / 'checkpoints/baselines/openvla_oft/openvla-7b-oft-finetuned-libero-spatial-object-goal-10'),
    output_dir=str(output))
configs = ROOT / 'run-configs'; configs.mkdir(exist_ok=True)
config_path = configs / 'openvla-oft-one-step-seed42.json'
config_path.write_text(json.dumps(config, indent=2))
run([OFT_PY, 'scripts/train/run_fft.py', '--model', 'openvla_oft', '--config', config_path, '--preflight'])
if not (output / 'step-0000001/COMPLETE').is_file():
    run([OFT_PY, 'scripts/train/run_fft.py', '--model', 'openvla_oft', '--config', config_path])
run([OFT_PY, 'scripts/train/check_fft_checkpoint.py', '--checkpoint', output / 'step-0000001',
     '--data', PREPARED, '--config', config_path, '--report', output / 'reload_report.json'])